# 第 03 章　从栅格到样本：构建训练表

> **本章问题**：怎样把一张张栅格，变成模型能读的一行行样本？

现在六张栅格已经对齐。对模型来说，图上的每个有效像元就是一个样本：有目标值，也有五个输入变量。我们要把这些像元摊成表格，同时把行列号和坐标留下来。为什么要留？因为模型算完以后，我们还得把预测值放回原来的地图位置。

**这一章结束时**，你会得到训练表、缺测统计，还会亲手把一组预测结果写回栅格。


## 3.1　先核对数据和运行环境

建模前先做个小检查：Notebook 是不是在 `geoml` 环境里？目标网格和边界能不能正常读？下面会打印解释器路径和 `CONDA_PREFIX`，以后复现时也能用它确认运行环境。


In [ ]:
import os
import sys
import time
import json
import tempfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import matplotlib.patheffects as pe
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import FormatStrFormatter, MaxNLocator

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
import geoml_data as gd
import geoml_models as gm
import geoml_style as gs

gs.setup()
T0 = time.perf_counter()
print("Python:", sys.executable)
print("Conda :", os.environ.get("CONDA_PREFIX"))

bundle = gd.load_stack()
bound = gpd.read_file(gd.RAW / "GanNan.shp")
print("目标网格 shape:", bundle["shape"])
print("CRS:", bundle["crs"])
print("transform:", tuple(round(v, 8) for v in bundle["transform"][:6]))
print("目标有效像元:", int(bundle["valid"].sum()))
print("边界要素数:", len(bound), "| 边界 CRS:", bound.crs)

目标网格有 106 行、176 列，其中 LST 有效像元是 7156 个。训练表就从这 7156 个目标像元开始，而不是把矩形网格外的 NoData 也塞进去。

注意，0.008983° 在这里约等于南北向 1 km、东西向 0.67 km。因此教程说“约 1 km 网格”只是方便交流，不表示像元真的是正方形。接下来我们把每个有效像元摊成一行。


In [ ]:
old_processed = gd.PROCESSED
with tempfile.TemporaryDirectory() as tmp:
    gd.PROCESSED = Path(tmp)
    try:
        table = gd.build_training_table(bundle)
    finally:
        gd.PROCESSED = old_processed

table.to_csv(gd.TABLES / "03_training_table.csv", index=False, encoding="utf-8-sig")
print("训练表形状:", table.shape)
display(table.head(8))

field_dict = pd.DataFrame([
    {"字段": "row", "含义": "目标网格行号（0 起，自北向南）", "单位": "像元",
     "范围": f"{int(table.row.min())} – {int(table.row.max())}"},
    {"字段": "col", "含义": "目标网格列号（0 起，自西向东）", "单位": "像元",
     "范围": f"{int(table.col.min())} – {int(table.col.max())}"},
    {"字段": "x", "含义": "像元中心经度", "单位": "度（EPSG:4326）",
     "范围": f"{table.x.min():.4f} – {table.x.max():.4f}"},
    {"字段": "y", "含义": "像元中心纬度", "单位": "度（EPSG:4326）",
     "范围": f"{table.y.min():.4f} – {table.y.max():.4f}"},
    {"字段": "lst", "含义": "目标变量：地表温度 LST", "单位": "原始像元值",
     "范围": f"{table.lst.min():.2f} – {table.lst.max():.2f}"},
    {"字段": "dem", "含义": "高程", "单位": "原始像元值",
     "范围": f"{table.dem.min():.1f} – {table.dem.max():.1f}"},
    {"字段": "ndvi", "含义": "归一化植被指数", "单位": "NDVI（无量纲）",
     "范围": f"{table.ndvi.min():.3f} – {table.ndvi.max():.3f}"},
    {"字段": "tem", "含义": "气温", "单位": "原始像元值",
     "范围": f"{table.tem.min():.2f} – {table.tem.max():.2f}"},
    {"字段": "precip", "含义": "年降水", "单位": "原始像元值",
     "范围": f"{table.precip.min():.1f} – {table.precip.max():.1f}"},
    {"字段": "soil", "含义": "土壤水分", "单位": "原始像元值",
     "范围": f"{table.soil.min():.2f} – {table.soil.max():.2f}"},
])
display(field_dict)

表头一共 10 列。`row/col` 是网格行列号，`x/y` 是像元中心坐标，`lst` 是目标，剩下五列是输入变量。

有个很容易犯的错误：把 `x/y` 也当作普通特征喂给模型。这样模型可能直接记住“哪里热”，在随机验证里分数很好看，换到没见过的区域就不灵。这里保留坐标是为了定位、写回地图和做空间诊断，不是拿来训练。


In [ ]:
FEATURES = gm.FEATURES
missing = table[FEATURES].isna().sum()
missing_pct = missing / len(table) * 100

miss_table = pd.DataFrame({
    "变量": [gd.FEATURE_LABELS[k] for k in FEATURES],
    "字段": FEATURES,
    "缺测像元": [int(missing[k]) for k in FEATURES],
    "缺测率(%)": [round(float(missing_pct[k]), 2) for k in FEATURES],
    "有效像元": [int(len(table) - missing[k]) for k in FEATURES],
})
miss_table.to_csv(gd.TABLES / "03_missingness.csv", index=False, encoding="utf-8-sig")
display(miss_table)

order = missing.sort_values(ascending=True).index
fig, ax = plt.subplots(figsize=(8.6, 3.8))
colors = ["#b23a48" if missing[k] > 0 else "#9aa0a6" for k in order]
ax.barh([gd.FEATURE_LABELS[k] for k in order], [missing[k] for k in order],
        color=colors, height=0.58, edgecolor="white")
for i, k in enumerate(order):
    v = int(missing[k])
    ax.text(v + 1.8, i, f"{v}（{missing_pct[k]:.2f}%）", va="center", fontsize=9.6, color=gs.INK)
ax.set_xlim(0, 120)
ax.set_xlabel("缺测像元数（分母：7156 个有效样本）")
ax.set_title("预测变量缺测统计：DEM 41、TEM 97，其余完整")
ax.grid(axis="x", color=gs.GRID, lw=0.6, alpha=0.9)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.text(0.01, -0.03, "缺测主要分布在县域边缘附近；具体成因需回到源数据核查。目标区外 NoData 不进入训练表，因此不计入分母。",
         fontsize=8.6, color="#555555")
gs.save(fig, "03_missingness.png")
plt.close(fig)

全表口径下，DEM 缺 41 个（0.57%），TEM 缺 97 个（1.36%），NDVI、降水、土壤水分完整。这个“完整”是相对于目标有效区说的：NDVI 源图在县域外有大量 0 填充，但对齐后目标区内没有整值 0，所以不能把 0 一律当缺测。

还有一个容易忽略的细节：缺测在空间折之间不是均匀分布的。只统计训练区 5796 行时，DEM/TEM 缺测是 38/91；独立测试区里 DEM 缺 3 个、TEM 缺 6 个（有重叠，合计 6 个像元至少缺一个变量）。若建模前直接 `dropna()`，训练集会少 99 行、测试区少 6 行，空间覆盖和最终评分口径都会变。正确做法是保留 `NaN`，把中位数填补和标准化放进 Pipeline，在每一折的训练部分单独 `fit`。

## 3.4 样本在地图上的什么位置

表里的每一行都对应地图上的一个像元。把 `lst` 放回 176×106 网格，就能看到样本的空间覆盖：彩色是 7156 个有效样本，浅灰是县域外 NoData。

In [ ]:
fig, ax = plt.subplots(figsize=(7.9, 5.2))
gs.plot_raster(
    ax, bundle["target"], bundle["extent"], "magma",
    title="有效样本的空间位置：7156 个像元，按 LST 着色",
    cbar_label="LST（原始像元值）",
    boundary=bound, crs=bundle["crs"], add_bar=True,
)
for artist in [*ax.texts, *ax.patches]:
    artist.set_path_effects([pe.withStroke(linewidth=2.8, foreground="white")])
for line in ax.lines[-3:]:
    line.set_path_effects([pe.withStroke(linewidth=3.2, foreground="white")])
ax.text(
    0.02, 0.965,
    "浅灰：目标区外 NoData\n每个彩色像元 = 训练表中的一行",
    transform=ax.transAxes, ha="left", va="top", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none", alpha=0.85),
    zorder=25,
)
gs.save(fig, "03_samples_map.png")
plt.close(fig)

样本覆盖了县域内所有有效像元，但分布并不均匀：边界形状不规则，西南和东南有整片 NoData；LST 从约 6.24 到 17.88（原始像元值），空间上呈连续高低分异，而不是随机噪点。这一点会直接决定第 04 章的验证策略——相邻像元太相似，随机抽点做验证会高估模型。

## 3.5 为什么填补和标准化必须在训练折内拟合

中位数、均值、标准差、最小最大值都是“从数据里估出来的参数”。如果在划分之前用全表算，验证折和测试区的信息就进入了预处理，模型在评分时已经“见过”它们。项目已经固定了 5 个空间折（第 04 章会解释它怎么来）。下面比较全表中位数与“去掉某一折后的训练部分”中位数：前者用了全部样本（错误），后者才是每折真正能看到的数。

In [ ]:
fold_table = gd.load_splits()
rows = [{
    "拟合数据": "全表（错误做法）",
    "DEM 中位数": table["dem"].median(),
    "TEM 中位数": table["tem"].median(),
}]
for f in range(gm.N_FOLDS):
    fit = fold_table[(fold_table["split"] == "train") & (fold_table["fold"] != f)]
    rows.append({
        "拟合数据": f"空间折 {f + 1} 的训练部分",
        "DEM 中位数": fit["dem"].median(),
        "TEM 中位数": fit["tem"].median(),
    })
median_check = pd.DataFrame(rows)
display(median_check.round(3))
print("DEM 折间极差:", round(median_check["DEM 中位数"].iloc[1:].max()
                            - median_check["DEM 中位数"].iloc[1:].min(), 3))
print("TEM 折间极差:", round(median_check["TEM 中位数"].iloc[1:].max()
                            - median_check["TEM 中位数"].iloc[1:].min(), 3))
print("岭回归 Pipeline 的步骤:", [name for name, _ in gm.ridge_model().steps])

全表中位数是 DEM 167.8、TEM 3.95；5 个空间折的训练部分中位数分别在 164.6–175.8 和 3.81–3.92 之间浮动，DEM 的折间极差 11.1（约 6.6%），TEM 为 0.11（约 2.9%）。差异看起来不大，但这不是“可以忽略”的理由：本例缺测率低，泄漏的影响小；换一个缺测集中、分布偏斜的变量，或者换用均值/分位数/协方差，偏差就会被放大。原则不变——**任何从特征或目标估出来的量，都只能在训练折内拟合**。

`gm.ridge_model()` 返回的是 `imputer → scaler → model` 的 Pipeline，`fit` 时中位数和均值只来自当前训练折，`predict` 时用同一套参数处理验证折。这就是本章要守住的结构。

## 3.6 把预测写回 GeoTIFF

模型最终的产物不是一张 PNG，而是一张能和原始栅格叠置的 GeoTIFF。写回时要带三样东西：原网格的 `transform`（像元起点与大小）、`CRS`、以及明确的 `NoData` 值。下面用“训练区 LST 均值”生成一张常数预测，走一遍完整流程：先构造完整网格（目标区外 `NaN`），写盘时把 `NaN` 变成 `-9999`，再读回检查几何和掩膜。

**这只是写回流程演示，不是正式模型。** 常数预测没有解释力，第 05 章才会比较真正的模型。

In [ ]:
splits = gd.load_splits()
train_mean = float(splits.loc[splits["split"] == "train", "lst"].mean())

full_grid = np.full(bundle["shape"], np.nan, dtype="float32")
full_grid[bundle["valid"]] = train_mean
pred_path = gd.write_prediction(full_grid, "03_demo_mean.tif")

with rasterio.open(pred_path) as ds:
    back = ds.read(1, masked=True)
    read_info = {
        "shape": (ds.height, ds.width),
        "transform": tuple(round(v, 8) for v in tuple(ds.transform)[:6]),
        "crs": ds.crs.to_string(),
        "nodata": ds.nodata,
        "dtype": ds.dtypes[0],
    }

back_arr = back.filled(np.nan).astype(float)
print("写回文件:", pred_path.relative_to(PROJECT))
print("读回元数据:", read_info)
print("有效像元:", int(np.isfinite(back_arr).sum()),
      "| NoData 像元:", int(np.ma.getmaskarray(back).sum()))
print("读回值范围:", float(np.nanmin(back_arr)), "–", float(np.nanmax(back_arr)))
print("有效掩膜与目标网格一致:", bool(np.array_equal(np.isfinite(back_arr), bundle["valid"])))

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.9), constrained_layout=True)
gs.plot_raster(
    axes[0], back_arr, bundle["extent"], "magma",
    vmin=float(table["lst"].min()), vmax=float(table["lst"].max()),
    title="(a) 写回的完整网格预测",
    cbar_label="LST 均值（原始像元值）",
    boundary=bound, crs=bundle["crs"], add_bar=True,
)
axes[0].text(
    0.02, 0.965,
    f"全图常数 = 训练区均值 {train_mean:.2f}\n目标区外写入 NoData = -9999",
    transform=axes[0].transAxes, ha="left", va="top", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none", alpha=0.85),
    zorder=25,
)

axes[1].set_facecolor(gs.MISSING_COLOR)
valid_img = np.where(np.isfinite(back_arr), 1.0, np.nan)
axes[1].imshow(
    np.ma.masked_invalid(valid_img), extent=bundle["extent"], origin="upper",
    cmap=ListedColormap(["#2a6f97"]), vmin=0, vmax=1,
    interpolation="nearest", aspect="auto", zorder=1,
)
bound.to_crs(bundle["crs"]).boundary.plot(ax=axes[1], color="#303030", lw=0.7, zorder=5)
axes[1].set_title("(b) 读回有效性：7156 个有效像元")
axes[1].xaxis.set_major_locator(MaxNLocator(5))
axes[1].yaxis.set_major_locator(MaxNLocator(5))
axes[1].xaxis.set_major_formatter(FormatStrFormatter("%.1f°E"))
axes[1].yaxis.set_major_formatter(FormatStrFormatter("%.1f°N"))
axes[1].legend(
    handles=[Patch(color="#2a6f97", label="有效预测"),
             Patch(color=gs.MISSING_COLOR, label="NoData = -9999")],
    loc="lower right", fontsize=8.5,
)
gs.add_north_arrow(axes[1])
gs.add_scalebar(axes[1], bundle["extent"])
fig.suptitle("把预测写回 GeoTIFF：流程演示，不是正式模型", fontsize=13.5, fontweight="semibold")
gs.save(fig, "03_writeback.png")
plt.close(fig)

读回结果与源网格逐项对上：`shape=(106, 176)`，`transform` 的六个参数与目标网格一致，`CRS=EPSG:4326`，`NoData=-9999.0`，有效像元 7156，读回值恒为训练区均值 11.95。右侧有效性图不是“模型效果”，而是证明 NoData 被正确写入：县域外是浅灰，县域内是完整覆盖。

到这里，栅格已经变成表；表还能变回栅格。接下来才轮到“用哪些像元训练、用哪些像元评分”。

## 地理视角

训练表的每一行都带着空间身份。`row/col` 让我们知道某个样本在网格中的位置，`x/y` 让它能回到经纬度；缺测不是随机撒点，而是沿着重采样边缘出现；LST 的高低也不是独立随机，而是连片分布。若只把表当成普通 DataFrame，很容易忘记：删掉一行，就是在地图上挖掉一个像元；随机划分，就是把相邻像元分到训练和验证两边。地理数据的不独立，是后面所有验证设计的起点。

## 常见坑

1. **先全表填补再划分**：中位数/均值/标准差都带了验证折信息。应把 `SimpleImputer`、`StandardScaler` 放进 Pipeline，在每折训练部分 `fit`。
2. **把 `row/col/x/y` 当特征**：模型会记住地理位置，随机 CV 虚高，换区域失效。它们只用于定位、诊断和写回。
3. **看到 NDVI=0 就全改 `NaN`**：目标区内没有整值 0，0 主要是县域外填充；应按掩膜判断，而不是按数值猜测。
4. **直接 `dropna()`**：会丢掉 105 个至少缺一个变量的样本（训练区 99、测试区 6），改变空间覆盖和最终评分口径。
5. **写回时手工拼 `transform`**：像元起点、符号、`NoData` 任一处写错，GIS 里就会整体错位或出现黑边。应直接复用目标网格元数据。

## 练习

用 `row/col` 把 DEM 缺测的 41 个像元和 TEM 缺测的 97 个像元画到 176×106 网格上（浅灰底、红色点），叠加县界，观察它们是否集中在边缘。然后回答：如果直接删除这些行，会主要损失县域的哪一部分？这是否会改变训练样本对全县的代表性？

## 小结

- 训练表是 7156 行 × 10 列：5 列位置/目标 + 5 个预测变量，缺测以 `NaN` 保留。
- DEM 缺 41 个（0.57%）、TEM 缺 97 个（1.36%），其余 3 个变量在目标区内完整；缺测沿重采样边缘分布。
- 任何填补、标准化、特征选择都只能在训练折内拟合；全表统计量会把验证信息带进模型。
- 预测写回必须复用目标网格的 `transform`、`CRS` 和显式 `NoData`；本章的常数预测只验证流程。
- 表里的每一行都能回到地图上的一个像元，空间位置是理解后续验证结果的关键。

In [ ]:
elapsed = time.perf_counter() - T0
fig_names = ["03_missingness.png", "03_samples_map.png", "03_writeback.png"]
run_summary = {
    "notebook": "03_构建训练表.ipynb",
    "kernel": "geoml",
    "python": sys.executable,
    "conda_prefix": os.environ.get("CONDA_PREFIX"),
    "运行时间_秒": round(elapsed, 1),
    "关键数字": {
        "训练表形状": list(table.shape),
        "目标有效像元": int(bundle["valid"].sum()),
        "DEM缺测": int(missing["dem"]),
        "TEM缺测": int(missing["tem"]),
        "其余变量缺测": {k: int(missing[k]) for k in ["ndvi", "precip", "soil"]},
        "经度范围": [round(float(table.x.min()), 4), round(float(table.x.max()), 4)],
        "纬度范围": [round(float(table.y.min()), 4), round(float(table.y.max()), 4)],
        "LST范围": [round(float(table.lst.min()), 2), round(float(table.lst.max()), 2)],
        "训练区LST均值": round(train_mean, 4),
        "写回有效像元": int(np.isfinite(back_arr).sum()),
        "写回NoData": float(read_info["nodata"]),
    },
    "图件": fig_names,
    "未核实事项": [
        "LST、TEM、降水、土壤水分单位元数据未注明，全文只写“原始像元值”",
        "LST/TEM 均为 2021 年，但月日与时间聚合方式未知",
        "降水/土壤水分为 2020 年，与 LST 不同期",
        "NDVI 无时间血缘，不能视为与 LST 同期观测",
        "案例定位为留出区域估计的方法演示，不是甘南县 LST 产品",
    ],
}
gd.write_json(run_summary, "03_run_summary.json")
print(f"本章执行耗时 {elapsed:.1f} 秒")
print("已保存:", ", ".join(fig_names), "03_run_summary.json")